# Medallion Architecture

> **A note for analysts:** You will mostly work with data already in the **Gold** layer. However, in this module **we replicate the architecture end-to-end as a learning experience** — so that when these terms come up in conversations or table names, you have a clear mental model of what they mean and how data got there.

## What is Medallion Architecture?

Medallion architecture is a **data design pattern** used to organise data in a lakehouse. It structures your data into three progressive layers — **Bronze**, **Silver**, and **Gold** — each representing a higher level of quality and refinement.

The core idea is simple: **raw data flows in at one end and clean, business-ready data comes out the other**.

```
  Source Systems  ──▶  Bronze (Raw)  ──▶  Silver (Cleaned)  ──▶  Gold (Business)
```

This layered approach gives you **reproducibility** (you always keep the raw data), **traceability** (you can track how data was transformed), and **flexibility** (different teams can work at different layers).

## How Can Medallion Architecture Be Implemented?

The important thing is **not where the data is stored, but the process and ordering of the stages**. Organisations implement the layers in different ways:

| Approach | Example | Notes |
| --- | --- | --- |
| **Catalog-level** | `catalog_30_bronze`, `catalog_20_silver`, `catalog_10_gold` | How the DfE implements it in production. Each layer is a separate catalog. |
| **Schema-level** | `my_catalog.bronze`, `my_catalog.silver`, `my_catalog.gold` | Layers are schemas within a single catalog. Simpler to set up; suitable for smaller projects or training. |
| **Naming convention** | `my_catalog.my_schema.bronze_students`, `...silver_students` | Layers are distinguished only by table name prefixes. Not recommended — harder to govern. |

For our learning exercises we use a **schema-level** approach within `catalog_40_copper_analyst_training`, with `bronze`, `silver`, and `gold` schemas.

## Bronze Layer – Raw Ingestion

| Aspect | Detail |
| --- | --- |
| **Purpose** | Land data exactly as it arrives from source systems |
| **Format** | Raw, unvalidated, append-only |
| **Typical content** | JSON payloads, CSV dumps, CDC streams, API responses |
| **Schema** | Matches the source; may include metadata columns (ingestion timestamp, source file name) |

**Key principles:**
* **No transformations** — store the data as-is so you have a complete audit trail
* **Append-only** — never overwrite; this protects you if upstream schemas change or bad data arrives
* **Metadata enrichment** — add columns like `_ingested_at`, `_source_file`, or `_batch_id` to aid debugging

## Silver Layer – Cleaned & Conformed

| Aspect | Detail |
| --- | --- |
| **Purpose** | Clean, deduplicate, validate, and conform the data |
| **Format** | Structured, typed, quality-checked |
| **Typical content** | Deduplicated records, joined reference data, standardised column names |
| **Schema** | Enforced and consistent; data types are correct |

**Key principles:**
* **Deduplication** — remove exact duplicates and apply business logic to resolve conflicts
* **Data quality checks** — filter or flag records that fail validation rules
* **Conformance** — standardise naming conventions, date formats, code lookups, and units of measure
* **Joins** — enrich records by joining with reference/dimension tables

## Gold Layer – Business-Ready

| Aspect | Detail |
| --- | --- |
| **Purpose** | Serve curated, aggregated / derived, business-level datasets |
| **Format** | Modelled, optimised for consumption |
| **Typical content** | KPI tables, summary statistics, analytical / dimensional models, feature stores |
| **Schema** | Tailored to specific business questions or reporting needs |

**Key principles:**
* **Business logic lives here** — aggregations, calculated metrics, and business rules are applied at this layer
* **Consumption-optimised** — tables are structured for dashboards, reports, ML models, or APIs
* **Multiple Gold tables from one Silver table** — different teams may create different Gold views of the same underlying data

## Copper Layer – Migrated Analytical Modelling Areas (DfE-Specific)

> **This layer is specific to the Department for Education and is not part of the standard medallion architecture.**

| Aspect | Detail |
| --- | --- |
| **Purpose** | Host pre-existing analytical modelling areas migrated to Databricks during onboarding |
| **Format** | As-is from the original source environment; structure and conventions vary |
| **Typical content** | Legacy analytical models, derived datasets, and reporting tables |
| **Schema** | Inherited from the original modelling area; not yet conformed to medallion standards |

**Why does Copper exist?**

When the Department onboarded analytical teams to Databricks, existing data and processes needed to be available **quickly** — without waiting for full remodelling into Bronze → Silver → Gold. Copper serves this transitional purpose.

**Key principles:**
* **Continuity** — analysts keep working with familiar datasets while migration is underway
* **Transitional by design** — the longer-term goal is to remodel Copper data into the standard medallion architecture
* **Pragmatic onboarding** — getting teams onto the platform quickly provides immediate benefits (collaboration, compute, governance)

**What does this mean for analysts?**

If you work with tables in a Copper schema, you are using migrated legacy data. It will work as expected, but over time these tables will be replaced by properly modelled Bronze → Silver → Gold equivalents. When that happens, you may need to update your queries to point to the new Gold layer tables.

## Putting It All Together

| | Bronze | Silver | Gold | Copper |
| --- | --- | --- | --- | --- |
| **Data quality** | Raw / as-is | Cleaned / validated | Business-ready | Varies (as migrated) |
| **Audience** | Data engineers | Data engineers / analysts | Analysts / stakeholders | Analysts (transitional) |
| **Update pattern** | Append-only | Merge / upsert | Rebuild / incremental | Inherited from source |
| **Longevity** | Permanent | Permanent | Permanent | Transitional |

### Why use it?

* **Recoverability** — if a transformation has a bug, you still have the raw Bronze data to reprocess from
* **Modularity** — each layer has a clear responsibility, making pipelines easier to maintain and debug
* **Governance** — you can apply different access controls at each layer (e.g. restrict Bronze to engineers, open Gold to analysts)
* **Incremental processing** — each layer can process only new/changed data rather than reprocessing everything

---

*This pattern is a guideline, not a rigid rule. Some organisations add additional layers (e.g. a "landing" zone before Bronze, or a "platinum" layer for ML features). The important thing is the principle: progressively refine your data from raw to business-ready.*


### ✅ Task: Check your Medallion architecture knowledge

Run the next chunk of code below and scroll down to take the Knowledge Check Quiz!

In [0]:
%python
# Run this cell to display the Knowledge Check Quiz
with open("/Workspace/Users/kimberly.hoskins@education.gov.uk/techskills_workshops/Databricks_workshops/Intro to Databricks and coding/01 Databricks fundamentals/quiz_medallion.html") as f:
    displayHTML(f.read())